In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
!nvidia-smi -L

GPU 0: Tesla T4 (UUID: GPU-6ecbceb4-440e-0f1d-d0ca-22c773aa2782)
GPU 1: Tesla T4 (UUID: GPU-34e91c97-6268-5e01-a9cb-a346ec9bf332)


In [2]:
!pip install -q laya chess

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.7/79.7 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 79.0 MB/s eta 0:00:00:00:010:01
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.5/214.5 kB 10.7 MB/s eta 0:00:00


In [3]:
!find /kaggle/input -name "*.jsonl" -o -name "*.py"

/kaggle/input/datasets/atavale/laya-chess-supporting-files/train_laya_chess.py
/kaggle/input/datasets/atavale/laya-chess-supporting-files/make_chess_dataset.py
/kaggle/input/datasets/atavale/laya-chess-supporting-files/laya_player.py
/kaggle/input/datasets/atavale/stockfish-chess-20k-true/chess_20k.jsonl
/kaggle/input/datasets/atavale/stockfish-chess-20k/chess_20k.jsonl


In [4]:
!cp /kaggle/input/datasets/atavale/stockfish-chess-20k-true/chess_20k.jsonl /kaggle/input/datasets/atavale/laya-chess-supporting-files/train_laya_chess.py /kaggle/input/datasets/atavale/laya-chess-supporting-files/laya_player.py /kaggle/working/

In [5]:
!cp -r /kaggle/input/datasets/atavale/laya-chess-20k-epoch-2/laya_chess_20k /kaggle/working/

In [12]:
!cd /kaggle/working && python train_laya_chess.py --data chess_20k.jsonl --out laya_smoke --smoke

Fetching 5 files: 100%|█████████████████████████| 5/5 [00:00<00:00, 6034.97it/s]
Download complete: : 0.00B [00:00, ?B/s]              
Base model: /root/.cache/huggingface/hub/models--convaiinnovations--laya/snapshots/55cf4c4ebb4ebe31b2550e8bdf3bd21b99753851 | max_len 512 | head_max_len 192 | device cuda | 1 process(es)
Positions: train 30, val 429, test 441
Training items: 120 | tokens per item: median 246, max 265 (limit 512)
Before training: {'positions': 5, 'top1_match': 0.4, 'avg_expected_score_lost': 0.16199999999999998, 'random_move_expected_score_lost': 0.21310567099567096} (7s)
Training 1 epoch(s), 120 items per process, 30 optimizer steps
/kaggle/working/train_laya_chess.py:356: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. Se

In [15]:
!cd /kaggle/working && torchrun --standalone --nproc_per_node=2 train_laya_chess.py --data chess_20k.jsonl --out laya_chess_20k --epochs 2 --eval-positions 300

W0927 20:27:35.846000 585 torch/distributed/run.py:852] 
W0927 20:27:35.846000 585 torch/distributed/run.py:852] *****************************************
W0927 20:27:35.846000 585 torch/distributed/run.py:852] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0927 20:27:35.846000 585 torch/distributed/run.py:852] *****************************************
[W927 20:27:36.469358901 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W927 20:27:38.289552969 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W927 20:27:38.376634998 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
Fetching 5 files: 100%|████████████████████████| 5/5 [00:00<00:00, 39346.19it/s]
Download complete: : 0.00B [00:00, ?B/s]              /usr/local/lib/p

In [16]:
!cd /kaggle/working && rm -rf laya_smoke && zip -qr laya_chess_20k.zip laya_chess_20k -x "laya_chess_20k/checkpoint_latest/*"

In [6]:
!ls /kaggle/working/laya_chess_20k

encoder  eval_report.json  model.safetensors  rl_agent_config.json  tokenizer


Updated learning rates to make it lower

In [7]:
!cd /kaggle/working && torchrun --standalone --nproc_per_node=2 train_laya_chess.py --data chess_20k.jsonl --base /kaggle/working/laya_chess_20k --out laya_chess_20k_round2 --epochs 3 --lr-encoder 1e-5 --lr-head 5e-5 --eval-positions 300

W0927 23:17:54.767000 139 torch/distributed/run.py:852] 
W0927 23:17:54.767000 139 torch/distributed/run.py:852] *****************************************
W0927 23:17:54.767000 139 torch/distributed/run.py:852] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0927 23:17:54.767000 139 torch/distributed/run.py:852] *****************************************
[W927 23:17:55.664436099 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W927 23:17:56.400421830 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W927 23:17:56.489381778 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
/usr/local/lib/python3.12/dist-packages/torch/distributed/c10d_logger.py:83: UserWarning: barrier(): using the device under current context. You can sp